In [ ]:
# CELL 1 - Install libraries
!pip install tensorflow scikit-learn matplotlib seaborn pillow

In [ ]:
# CELL 2 - Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted")

In [ ]:
# CELL 3 - Unzip dataset from Google Drive
# Upload archive__1_.zip to your Google Drive first
!unzip /content/drive/MyDrive/archive__1_.zip -d /content/
print("Dataset unzipped")

In [ ]:
# CELL 4 - Import libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils import shuffle
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

print("Libraries imported")
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# CELL 5 - Settings
IMAGE_SIZE = 128
BATCH_SIZE = 32
EPOCHS     = 3

train_dir = "/content/Training/"
test_dir  = "/content/Testing/"

print("Settings done")
print("Train dir:", train_dir)
print("Test dir:", test_dir)
print("Classes:", os.listdir(train_dir))

In [ ]:
# CELL 6 - Load dataset
train_datagen = ImageDataGenerator(
    rescale          = 1./255,
    rotation_range   = 20,
    horizontal_flip  = True,
    brightness_range = [0.8, 1.2],
    zoom_range       = 0.2,
    validation_split = 0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size = (IMAGE_SIZE, IMAGE_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = "sparse",
    subset      = "training"
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size = (IMAGE_SIZE, IMAGE_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = "sparse",
    subset      = "validation"
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size = (IMAGE_SIZE, IMAGE_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = "sparse"
)

class_names = list(train_generator.class_indices.keys())

print("Train images:", train_generator.samples)
print("Val images:", val_generator.samples)
print("Test images:", test_generator.samples)
print("Classes:", class_names)

In [ ]:
# CELL 7 - Show sample images
images, labels = next(train_generator)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()
for i in range(10):
    axes[i].imshow(images[i])
    axes[i].axis("off")
    axes[i].set_title(class_names[int(labels[i])])
plt.tight_layout()
plt.show()
print("Sample images shown")

In [ ]:
# CELL 8 - Build model with VGG16
base_model = VGG16(
    input_shape = (IMAGE_SIZE, IMAGE_SIZE, 3),
    include_top = False,
    weights     = "imagenet"
)

# Freeze all VGG16 layers
for layer in base_model.layers:
    layer.trainable = False

model = Sequential()
model.add(Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
model.add(base_model)
model.add(Flatten())
model.add(Dropout(0.5))
model.add(Dense(128, activation="relu", kernel_regularizer=l2(0.001)))
model.add(Dropout(0.4))
model.add(Dense(len(class_names), activation="softmax"))

model.compile(
    optimizer = Adam(learning_rate=0.0001),
    loss      = "sparse_categorical_crossentropy",
    metrics   = ["accuracy"]
)

model.summary()
print("Model built")

In [ ]:
# CELL 9 - Train the model
callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)
]

print("Training started...")

history = model.fit(
    train_generator,
    epochs          = EPOCHS,
    validation_data = val_generator,
    callbacks       = callbacks
)

print("Training done")
print("Train Accuracy:", round(history.history["accuracy"][-1] * 100, 2), "%")
print("Val Accuracy:", round(history.history["val_accuracy"][-1] * 100, 2), "%")

In [ ]:
# CELL 10 - Evaluate on test data
print("Evaluating on test data...")
results = model.evaluate(test_generator)
print("Test Loss:", round(results[0], 4))
print("Test Accuracy:", round(results[1] * 100, 2), "%")

In [ ]:
# CELL 11 - Classification Report
test_generator.reset()
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# CELL 12 - Plot accuracy and loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["accuracy"],     label="Train Accuracy")
axes[0].plot(history.history["val_accuracy"], label="Val Accuracy", linestyle="--")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history["loss"],     label="Train Loss",  color="tomato")
axes[1].plot(history.history["val_loss"], label="Val Loss",    color="orange", linestyle="--")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()
print("Plots shown")

In [ ]:
# CELL 13 - Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
print("Confusion matrix shown")

In [ ]:
# CELL 14 - Save model to Google Drive
model.save("/content/drive/MyDrive/brain_tumor_model.h5")
print("Model saved to Google Drive")

In [ ]:
# CELL 15 - Predict a single image
def detect_and_display(img_path, model, image_size=128):
    img       = load_img(img_path, target_size=(image_size, image_size))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions         = model.predict(img_array)
    predicted_class_idx = np.argmax(predictions, axis=1)[0]
    confidence          = np.max(predictions, axis=1)[0]

    if class_names[predicted_class_idx] == "notumor":
        result = "No Tumor"
    else:
        result = "Tumor: " + class_names[predicted_class_idx]

    plt.imshow(load_img(img_path))
    plt.axis("off")
    plt.title(result + " (Confidence: " + str(round(confidence * 100, 2)) + "%)")
    plt.show()
    print("Prediction:", result)
    print("Confidence:", round(confidence * 100, 2), "%")

# Upload any image to Google Drive and update path below
image_path = "/content/drive/MyDrive/your_image.jpg"
detect_and_display(image_path, model)